# CLARITY-SemEval 2026 · FEATURE ENGINEERING



## Imports

In [23]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from textwrap import shorten
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

sns.set_theme(style='whitegrid', context='talk')
pd.options.display.max_columns = 20

import pandas as pd

pd.set_option("display.max_colwidth", None)   # kolonları kesme
pd.set_option("display.width", 200)          # satır genişliğini büyüt
pd.set_option("display.max_columns", None)   # tüm kolonları göster


## Load the QEvasion Dataset

Attempts to read the cached CSV under `../data/raw`. If it does not exist locally the HuggingFace copy is downloaded (matching the repo instructions).

In [22]:
# RAW QEvasion from Hugging Face (no reordering, no dropping, no preprocessing)

from datasets import load_dataset
import pandas as pd

HF_DATASET = "ailsntua/QEvasion"

# 1) Load HF dataset as-is
ds = load_dataset(HF_DATASET)

print("Available splits:", list(ds.keys()))
for split_name in ds.keys():
    print(f"{split_name} rows:", len(ds[split_name]))

# 2) Convert splits to pandas (RAW)
train_raw = ds["train"].to_pandas()
test_raw  = ds["test"].to_pandas() if "test" in ds else None

print("\n=== RAW TRAIN COLUMNS ===")
print(list(train_raw.columns))

print("\n=== RAW TRAIN HEAD (ALL COLUMNS) ===")
display(train_raw.head(2))




Available splits: ['train', 'test']
train rows: 3448
test rows: 308

=== RAW TRAIN COLUMNS ===
['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label']

=== RAW TRAIN HEAD (ALL COLUMNS) ===


,title,date,president,url,question_order,interview_question,interview_answer,gpt3.5_summary,gpt3.5_prediction,question,annotator_id,annotator1,annotator2,annotator3,inaudible,multiple_questions,affirmative_questions,index,clarity_label,evasion_label
0,"The President's News Conference in Hanoi, Vietnam","September 10, 2023",Joseph R. Biden,https://www.presidency.ucsb.edu/documents/the-presidents-news-conference-hanoi-vietnam-0,1,Q. Of the Biden administration. And accused the United States of containing China while pushing for diplomatic talks.How would you respond to that? And do you think President Xi is being sincere about getting the relationship back on track as he bans Apple in China?,"Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues.And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things.And so, really, what this trip was aboutit was less about containing China. I don't want to contain China. I just want to make sure that we have a relationship with China that is on the up and up, squared away, everybody knows what it's all about. And one of the ways you do that is, you make sure that we are talking about the same things.And I think that one of the things we've doneI've tried to do, and I've talked with a number of my staff about this for the last, I guess, 6 monthsis, we have an opportunity to strengthen alliances around the world to maintain stability.That's what this trip was all about: having India cooperate much more with the United States, be closer with the United States, Vietnam being closer with the United States. It's not about containing China; it's about having a stable base, a stable base in the Indo-Pacific.And it'sfor example, when I was spending a lot of time talking with President Xi, he asked why we were doingwhy was I going to have the Quad, meaning Australia, India, Japan, and the United States? And I said, To maintain stability. It's not about isolating China. It's about making sure the rules of the roadeverything from airspace and space in the ocean isthe international rules of the road are abided by.And soand I hope thatI think that Prime Minister XiI mean, Xi has somesome difficulties right now. All countries end up with difficulties, and he had some economic difficulties he's working his way through. I want to see China succeed economically, but I want to see them succeed by the rules.The next question was to Bloomberg.","The question consists of 2 parts: \n1. How would you respond to the accusation that the United States is containing China while pushing for diplomatic talks?\n2. Do you think President Xi is being sincere about getting the relationship back on track as he bans Apple in China?\n\nThe response provides the following information regarding these points:\n1. The President expresses sincerity about getting the relationship between the United States and China right.\n2. China is changing some of the rules of the game, such as banning Chinese government officials from using Western cell phones.\n3. The purpose of the trip was not to contain China but to establish a stable relationship with China and strengthen alliances in the Indo-Pacific region.\n4. The Quad (Australia, India, Japan, and United States) is not meant to isolate China but to maintain stability and ensure that international rules are followed.\n5. President Xi has some economic difficulties, and the President hopes to see China succeed economically while also following the rules.",Question part: 1. How would you respond to the accusation that the United States is containing China while pushing for diplomatic talks?\nVerdict: 1.1 Explicit - The information requested is explicitly stated (in the requested form)\

In [21]:
if test_raw is not None:
    print("\n=== RAW TEST COLUMNS ===")
    print(list(test_raw.columns))
    print("\n=== RAW TEST HEAD (ALL COLUMNS) ===")
    display(test_raw.head(3))


=== RAW TEST COLUMNS ===
['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label']

=== RAW TEST HEAD (ALL COLUMNS) ===


,title,date,president,url,question_order,interview_question,interview_answer,gpt3.5_summary,gpt3.5_prediction,question,annotator_id,annotator1,annotator2,annotator3,inaudible,multiple_questions,affirmative_questions,index,clarity_label,evasion_label
0,None,None,None,https://www.presidency.ucsb.edu/documents/the-presidents-news-conference-3,5,"Q. What about the redline, sir?","Well, the world has made it clear that these tests caused us to come together and work in the United Nations to send a clear message to the North Korean regime. We're bound up together with a common strategy to solve this issue peacefully through diplomatic means.Kevin [Kevin Corke, NBC News].",None,None,Inquiring about the status or information regarding the redline.,None,Dodging,General,Dodging,False,False,True,0,Ambivalent,
1,None,None,None,https://www.presidency.ucsb.edu/documents/the-presidents-news-conference-with-president-lee-myung-bak-south-korea,2,Q. Will you invite them to the White House to negotiate on the jobs bill?,"I think that anytime and anyplace that they are serious about working on putting people back to work, we'll be prepared to work with them. But we're not going to create a lot of theater that then results in them engaging in the usual political talking points, but don't result in action.People want action. And I'm prepared to work with them. But again, the last time I was here at a press conference I said--I asked you guys to show us the Republican jobs plan that independent economists would indicate would actually put people back to work. I haven't yet seen it. And so eventually, I'm hoping that they actually put forward some proposals that indicate that they feel that sense of urgency about people--needing to put people back to work right now.All right, Jessica, you can't have four follow-ups. One is good. All right.",None,None,Will you invite them to the White House to negotiate on the jobs bill?,None,Deflection,General,General,False,False,False,1,Ambivalent,
2,None,None,None,https://www.presidency.ucsb.edu/documents/the-presidents-news-conference-chicago,1,"Q. Harsh. Mr. President, Japan has dropped the threat of sanctions from its proposed Security Council resolution about North Korea. Why was that necessary? And how do you punish or penalize a country that's already among the poorest and most isolated in the world?","I think that the purpose of the U.N. Security Council resolution is to send a clear message to the leader of that the world condemns that which he did. Part of our strategy, as you know, has been to have others at the table, is to say as clearly as possible to the n, Get rid of your weapons, and there's a better way forward. In other words, there's a choice for him to make. He can verifiably get rid of his weapons programs and stop testing rockets, and there's a way forward for him to help his people.I believe it's best to make that choice clear to him with more than one voice, and that's why we have the six-party talks. And now that he has defied China and Japan and South Korea and Russia and the United States—all of us said, don't fire that rocket. He not only fired one; he fired seven. Now that he made that defiance, it's best for all of us to go to the U.N. Security Council and say loud and clear, here are some red lines. And that's what we're in the process of doing.The problem with diplomacy, it takes a while to get something done. If you're acting alone, you can move quickly. When you're rallying world opinion and trying to come up with the right language at the United Nations to send a clear signal, it takes a while.And so yesterday I was on the phone with—I think I mentioned this to the press conference yesterday—to Hu Jintao and Vladimir Putin; the day before to President Roh and Prime Minister Koizumi. And Condi, by the way, was making the same calls out there to her counterparts, all aiming at saying, It's your choice, Kim Jong Il; you've got the choice to make.So we'll see what happens at the U.N. 

In [24]:
# =========================
# QEvasion (HF) -> Load + Clean Columns
# =========================

from pathlib import Path
import pandas as pd
from datasets import load_dataset

# -------------------------
# Configuration
# -------------------------
HF_DATASET = "ailsntua/QEvasion"
CACHE_DIR = Path("./hf_cache")
OUT_DIR = Path("./data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = OUT_DIR / "QEvasion_train_clean.csv"
TEST_CSV  = OUT_DIR / "QEvasion_test_clean.csv"

# -------------------------
# 1) Load EXACTLY as published on Hugging Face
# -------------------------
ds = load_dataset(HF_DATASET, cache_dir=str(CACHE_DIR))

print("Available splits:", list(ds.keys()))
for split_name in ds.keys():
    print(f"Rows in {split_name}: {len(ds[split_name]):,}")

# -------------------------
# 2) Convert to pandas
# -------------------------
train_df = ds["train"].to_pandas()
test_df  = ds["test"].to_pandas() if "test" in ds else None

print("\n=== ORIGINAL TRAIN columns ===")
print(list(train_df.columns))

# -------------------------
# 3) Drop unwanted columns
# -------------------------
cols_to_drop = [
    "gpt3.5_summary",
    "annotator_id",
    "annotator1",
    "annotator2",
    "annotator3",
    "question_order",
    "inaudible",
    "affirmative_questions",
    "multiple_questions",
    "date",
    "url",
    "index",
]

# Train
drop_train = [c for c in cols_to_drop if c in train_df.columns]
train_df_clean = train_df.drop(columns=drop_train)

# Test
if test_df is not None:
    drop_test = [c for c in cols_to_drop if c in test_df.columns]
    test_df_clean = test_df.drop(columns=drop_test)
else:
    test_df_clean = None

# -------------------------
# 4) Optional: simple reorder of remaining main columns
#    (others, if any, are appended at the end)
# -------------------------
desired_order = [
    "title",
    "president",
    "interview_question",
    "interview_answer",
    "clarity_label",
    "evasion_label",
    "gpt3.5_prediction",
    "question",
]

def reorder_simple(df: pd.DataFrame) -> pd.DataFrame:
    ordered = [c for c in desired_order if c in df.columns]
    rest = [c for c in df.columns if c not in ordered]
    return df[ordered + rest]

train_df_clean = reorder_simple(train_df_clean)
if test_df_clean is not None:
    test_df_clean = reorder_simple(test_df_clean)

# -------------------------
# 5) Save & preview
# -------------------------
train_df_clean.to_csv(TRAIN_CSV, index=False)
if test_df_clean is not None:
    test_df_clean.to_csv(TEST_CSV, index=False)

print("\n=== CLEAN TRAIN columns ===")
print(list(train_df_clean.columns))
print("\nTRAIN CLEAN PREVIEW:")
display(train_df_clean.head(5))




Available splits: ['train', 'test']
Rows in train: 3,448
Rows in test: 308

=== ORIGINAL TRAIN columns ===
['title', 'date', 'president', 'url', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'index', 'clarity_label', 'evasion_label']

=== CLEAN TRAIN columns ===
['title', 'president', 'interview_question', 'interview_answer', 'clarity_label', 'evasion_label', 'gpt3.5_prediction', 'question']

TRAIN CLEAN PREVIEW:


,title,president,interview_question,interview_answer,clarity_label,evasion_label,gpt3.5_prediction,question
0,"The President's News Conference in Hanoi, Vietnam",Joseph R. Biden,Q. Of the Biden administration. And accused the United States of containing China while pushing for diplomatic talks.How would you respond to that? And do you think President Xi is being sincere about getting the relationship back on track as he bans Apple in China?,"Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues.And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things.And so, really, what this trip was aboutit was less about containing China. I don't want to contain China. I just want to make sure that we have a relationship with China that is on the up and up, squared away, everybody knows what it's all about. And one of the ways you do that is, you make sure that we are talking about the same things.And I think that one of the things we've doneI've tried to do, and I've talked with a number of my staff about this for the last, I guess, 6 monthsis, we have an opportunity to strengthen alliances around the world to maintain stability.That's what this trip was all about: having India cooperate much more with the United States, be closer with the United States, Vietnam being closer with the United States. It's not about containing China; it's about having a stable base, a stable base in the Indo-Pacific.And it'sfor example, when I was spending a lot of time talking with President Xi, he asked why we were doingwhy was I going to have the Quad, meaning Australia, India, Japan, and the United States? And I said, To maintain stability. It's not about isolating China. It's about making sure the rules of the roadeverything from airspace and space in the ocean isthe international rules of the road are abided by.And soand I hope thatI think that Prime Minister XiI mean, Xi has somesome difficulties right now. All countries end up with difficulties, and he had some economic difficulties he's working his way through. I want to see China succeed economically, but I want to see them succeed by the rules.The next question was to Bloomberg.",Clear Reply,Explicit,Question part: 1. How would you respond to the accusation that the United States is containing China while pushing for diplomatic talks?\nVerdict: 1.1 Explicit - The information requested is explicitly stated (in the requested form)\nExplanation: The President directly responds to the accusation by stating that the purpose of the trip and their approach is not about containing China but about establishing a stable relationship and ensuring that both countries are on the same page.\n\nQuestion part: 2. Do you think President Xi is being sincere about getting the relationship back on track as he bans Apple in China?\nVerdict: 1.1 Explicit - The information requested is explicitly stated (in the requested form)\nExplanation: The President explicitly expresses their belief in President Xi's sincerity about getting the relationship back on track while also mentioning the difficulties China is facing and their hope that China follows the rules.,How would you respond to the accusation that the United States is containing China while pushing for diplomatic talks?
1,"The President's News Conference in Hanoi, Vietnam",Joseph R. Biden,Q. Of the Biden administration. And accused the United States of containing China while pushing for diplomatic talks.How would you respond to that? And do you think President Xi is being sincere about getting the relationship back on track as he bans Apple in China?,"Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is g

In [ ]:
if test_df_clean is not None:
    print("\nTEST CLEAN PREVIEW:")
    display(test_df_clean.head(5))

**Stopword listeleri + tokenize**

In [2]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOP_SK = set(ENGLISH_STOP_WORDS)

def tokenize(text: str):
    """Lowercase + basit kelime tokenizasyonu."""
    if not isinstance(text, str):
        return []
    text = text.lower()
    return re.findall(r"[a-z']+", text)

def stopword_ratio(tokens, stop_set=STOP_SK):
    """tokens içindeki stopword oranı."""
    if not tokens:
        return 0.0
    stop_count = sum(1 for t in tokens if t in stop_set)
    return stop_count / len(tokens)


In [3]:
def compute_stopword_features_row(row):
    q_text = row.get("interview_question", "")
    a_text = row.get("interview_answer", "")

    q_tokens = tokenize(q_text)
    a_tokens = tokenize(a_text)

    q_ratio = stopword_ratio(q_tokens)
    a_ratio = stopword_ratio(a_tokens)
    diff   = a_ratio - q_ratio  # cevap soruya göre ne kadar daha stopword'lü?

    return pd.Series({
        "q_stop_ratio": q_ratio,
        "a_stop_ratio": a_ratio,
        "stop_ratio_diff": diff,
    })


In [5]:
train_with_stop = train_df_reordered.copy()

train_with_stop[["q_stop_ratio", "a_stop_ratio", "stop_ratio_diff"]] = (
    train_with_stop.apply(compute_stopword_features_row, axis=1)
)

# Soru ve cevap token listeleri
train_with_stop["q_tokens"] = train_with_stop["interview_question"].apply(tokenize)
train_with_stop["a_tokens"] = train_with_stop["interview_answer"].apply(tokenize)

# Hangi tokenların stopword olduğunu listeleyen kolonlar
train_with_stop["q_stop_tokens"] = train_with_stop["q_tokens"].apply(
    lambda toks: [t for t in toks if t in STOP_SK]
)
train_with_stop["a_stop_tokens"] = train_with_stop["a_tokens"].apply(
    lambda toks: [t for t in toks if t in STOP_SK]
)


cols_view = [

    "interview_question",
    "interview_answer",
    "q_tokens",
    "a_tokens",
    "q_stop_tokens",
    "q_stop_ratio",
    "a_stop_tokens",
    "a_stop_ratio",
    "stop_ratio_diff",
    "clarity_label",
    "evasion_label",
]

train_with_stop[cols_view].head()


NameError: name 'train_df_reordered' is not defined

In [ ]:
if test_df_reordered is not None:
    test_with_stop = test_df_reordered.copy()
    test_with_stop[["q_stop_ratio", "a_stop_ratio", "stop_ratio_diff"]] = (
        test_with_stop.apply(compute_stopword_features_row, axis=1)
    )

    test_with_stop["q_tokens"] = test_with_stop["interview_question"].apply(tokenize)
    test_with_stop["a_tokens"] = test_with_stop["interview_answer"].apply(tokenize)

    test_with_stop["q_stop_tokens"] = test_with_stop["q_tokens"].apply(
        lambda toks: [t for t in toks if t in STOP_SK]
    )
    test_with_stop["a_stop_tokens"] = test_with_stop["a_tokens"].apply(
        lambda toks: [t for t in toks if t in STOP_SK]
    )

test_with_stop[cols_view].head()


,interview_question,interview_answer,q_tokens,a_tokens,q_stop_tokens,q_stop_ratio,a_stop_tokens,a_stop_ratio,stop_ratio_diff,clarity_label,evasion_label
0,"Q. What about the redline, sir?","Well, the world has made it clear that these tests caused us to come together and work in the United Nations to send a clear message to the North Korean regime. We're bound up together with a common strategy to solve this issue peacefully through diplomatic means.Kevin [Kevin Corke, NBC News].","[q, what, about, the, redline, sir]","[well, the, world, has, made, it, clear, that, these, tests, caused, us, to, come, together, and, work, in, the, united, nations, to, send, a, clear, message, to, the, north, korean, regime, we're, bound, up, together, with, a, common, strategy, to, solve, this, issue, peacefully, through, diplomatic, means, kevin, kevin, corke, nbc, news]","[what, about, the]",0.500000,"[well, the, has, made, it, that, these, us, to, together, and, in, the, to, a, to, the, up, together, with, a, to, this, through]",0.461538,-0.038462,Ambivalent,
1,Q. Will you invite them to the White House to negotiate on the jobs bill?,"I think that anytime and anyplace that they are serious about working on putting people back to work, we'll be prepared to work with them. But we're not going to create a lot of theater that then results in them engaging in the usual political talking points, but don't result in action.People want action. And I'm prepared to work with them. But again, the last time I was here at a press conference I said--I asked you guys to show us the Republican jobs plan that independent economists would indicate would actually put people back to work. I haven't yet seen it. And so eventually, I'm hoping that they actually put forward some proposals that indicate that they feel that sense of urgency about people--needing to put people back to work right now.All right, Jessica, you can't have four follow-ups. One is good. All right.","[q, will, you, invite, them, to, the, white, house, to, negotiate, on, the, jobs, bill]","[i, think, that, anytime, and, anyplace, that, they, are, serious, about, working, on, putting, people, back, to, work, we'll, be, prepared, to, work, with, them, but, we're, not, going, to, create, a, lot, of, theater, that, then, results, in, them, engaging, in, the, usual, political, talking, points, but, don't, result, in, action, people, want, action, and, i'm, prepared, to, work, with, them, but, again, the, last, time, i, was, here, at, a, press, conference, i, said, i, asked, you, guys, to, show, us, the, republican, jobs, plan, that, independent, economists, would, indicate, would, actually, put, people, back, to, work, i, ...]","[will, you, them, to, the, to, on, the, bill]",0.600000,"[i, that, and, that, they, are, serious, about, on, back, to, be, to, with, them, but, not, to, a, of, that, then, in, them, in, the, but, in, and, to, with, them, but, again, the, last, i, was, here, at, a, i, i, you, to, show, us, the, that, would, would, put, back, to, i, yet, it, and, so, that, they, put, some, that, that, they, that, of, about, to, put, back, to, now, all, you, have, four, one, is, all]",0.540000,-0.060000,Ambivalent,
2,"Q. Harsh. Mr. President, Japan has dropped the threat of sanctions from its proposed Security Council resolution about North Korea. Why was that necessary? And how do you punish or penalize a country that's already among the poorest and most isolated in the world?","I think that the purpose of the U.N. Security Council resolution is to send a clear message to the leader of that the world condemns that which he did. Part of our strategy, as you know, has been to have others at the table, is to say as clearly as possible to the n, Get rid of your weapons, and there's a better way forward. In other words, there's a choice for him to make. He can verifiably get rid of his weapons programs and stop testing rockets, and there's a way forward for him to help his people.I believe it's b

In [ ]:
train_with_stop.to_csv(OUT_DIR / "QEvasion_train_with_stop.csv", index=False)
if test_df_reordered is not None:
    test_with_stop.to_csv(OUT_DIR / "QEvasion_test_with_stop.csv", index=False)


In [ ]:
print("TRAIN shape:", train_with_stop.shape)
print("TRAIN columns:", list(train_with_stop.columns))


TRAIN shape: (3448, 27)
TRAIN columns: ['title', 'president', 'question_order', 'interview_question', 'interview_answer', 'gpt3.5_summary', 'gpt3.5_prediction', 'question', 'clarity_label', 'evasion_label', 'annotator_id', 'annotator1', 'annotator2', 'annotator3', 'inaudible', 'multiple_questions', 'affirmative_questions', 'date', 'url', 'index', 'q_stop_ratio', 'a_stop_ratio', 'stop_ratio_diff', 'q_tokens', 'a_tokens', 'q_stop_tokens', 'a_stop_tokens']


In [ ]:
cols_view = [
    "interview_question",
    "interview_answer",
    "q_tokens",
    "a_tokens",
    "q_stop_tokens",
    "q_stop_ratio",
    "a_stop_tokens",
    "a_stop_ratio",
    "stop_ratio_diff",
    "clarity_label",
    "evasion_label",
]

train_with_stop[cols_view].head(5)


,interview_question,interview_answer,q_tokens,a_tokens,q_stop_tokens,q_stop_ratio,a_stop_tokens,a_stop_ratio,stop_ratio_diff,clarity_label,evasion_label
0,Q. Of the Biden administration. And accused the United States of containing China while pushing for diplomatic talks.How would you respond to that? And do you think President Xi is being sincere about getting the relationship back on track as he bans Apple in China?,"Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues.And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things.And so, really, what this trip was aboutit was less about containing China. I don't want to contain China. I just want to make sure that we have a relationship with China that is on the up and up, squared away, everybody knows what it's all about. And one of the ways you do that is, you make sure that we are talking about the same things.And I think that one of the things we've doneI've tried to do, and I've talked with a number of my staff about this for the last, I guess, 6 monthsis, we have an opportunity to strengthen alliances around the world to maintain stability.That's what this trip was all about: having India cooperate much more with the United States, be closer with the United States, Vietnam being closer with the United States. It's not about containing China; it's about having a stable base, a stable base in the Indo-Pacific.And it'sfor example, when I was spending a lot of time talking with President Xi, he asked why we were doingwhy was I going to have the Quad, meaning Australia, India, Japan, and the United States? And I said, To maintain stability. It's not about isolating China. It's about making sure the rules of the roadeverything from airspace and space in the ocean isthe international rules of the road are abided by.And soand I hope thatI think that Prime Minister XiI mean, Xi has somesome difficulties right now. All countries end up with difficulties, and he had some economic difficulties he's working his way through. I want to see China succeed economically, but I want to see them succeed by the rules.The next question was to Bloomberg.","[q, of, the, biden, administration, and, accused, the, united, states, of, containing, china, while, pushing, for, diplomatic, talks, how, would, you, respond, to, that, and, do, you, think, president, xi, is, being, sincere, about, getting, the, relationship, back, on, track, as, he, bans, apple, in, china]","[well, look, first, of, all, thei, am, sincere, about, getting, the, relationship, right, and, one, of, the, things, that, is, going, on, now, is, china, is, beginning, to, change, some, of, the, rules, of, the, game, in, terms, of, trade, and, other, issues, and, so, one, of, the, things, we, talked, about, for, example, is, that, they're, now, talking, about, making, sure, that, no, chineseno, one, in, the, chinese, government, can, use, a, western, cell, phone, those, kinds, of, things, and, so, really, what, this, trip, was, aboutit, was, less, about, containing, china, i, don't, want, to, contain, china, i, ...]","[of, the, and, the, of, while, for, how, would, you, to, that, and, do, you, is, being, sincere, about, the, back, on, as, he, in]",0.543478,"[well, first, of, all, am, sincere, about, the, and, one, of, the, that, is, on, now, is, is, to, some, of, the, of, the, in, of, and, other, and, so, one, of, the, we, about, for, is, that, now, about, that, no, one, in, the, can, a, those, of, and, so, what, this, was, was, less, about, i, to, i, to, that, we, have, a, with, that, is, on, the, up, and, up, what, all, about, and, one, of, the, you, do, that, is, you, that, we, are, about, the, same, and, i, that, one, of, the, 

In [ ]:
train_with_stop[cols_view].sample(1, random_state=42)


,interview_question,interview_answer,q_tokens,a_tokens,q_stop_tokens,q_stop_ratio,a_stop_tokens,a_stop_ratio,stop_ratio_diff,clarity_label,evasion_label
2900,"Q. I have a question for President Lee. Korea and the United States have made many achievements through the summit meeting, especially North Korean nuclear issue and the strengthening of the alliance. As for North Korean nuclear issue, Mr. President Lee suggested setting up a permanent liaison office in both Seoul and Pyongyang. What are some of the follow-up effects, if you do have any follow-up actions? And do you have any thoughts of proposing a meeting with Chairman Kim at an earlier date?","The process is not something that we discussed between ourselves during the summit meeting. In fact, when I was staying in Washington, DC, I had an interview with one of the newspapers there, and it came up. Of course, it was not a sudden suggestion. I did have a meeting among my staff and related ministries, and I talked about this in detail before I came to the United States.We have a new administration in Korea, and we haven't yet to begun dialogue with the ns. Inter-Korean dialogue, there is a need for us to have dialogue all the time. Up until now, we had dialogues whenever the need arose, and then it would stop. However, dialogue should be based on genuine cooperation and sincerity. And so with this in mind, I thought that it would be helpful to set up a permanent liaison office in both Seoul and Pyongyang.As for the summit meeting between myself and Chairman Kim, I will agree to it when the need is real. And I already said publicly that I am willing to meet with him—not just once, but many times—but if the meeting will yield substantial and real results. I believe only when that is possible, I am ready to meet with him and have sincere dialogue, because that will help to bring about peace and stability of the peninsula.So basically, I do hold that thought, but I'm not suggesting that—to have a meeting with Chairman Kim anytime soon. If the need arises, again, I'm ready to meet with him. Steven Lee [Steven Lee Myers, New York Times].","[q, i, have, a, question, for, president, lee, korea, and, the, united, states, have, made, many, achievements, through, the, summit, meeting, especially, north, korean, nuclear, issue, and, the, strengthening, of, the, alliance, as, for, north, korean, nuclear, issue, mr, president, lee, suggested, setting, up, a, permanent, liaison, office, in, both, seoul, and, pyongyang, what, are, some, of, the, follow, up, effects, if, you, do, have, any, follow, up, actions, and, do, you, have, any, thoughts, of, proposing, a, meeting, with, chairman, kim, at, an, earlier, date]","[the, process, is, not, something, that, we, discussed, between, ourselves, during, the, summit, meeting, in, fact, when, i, was, staying, in, washington, dc, i, had, an, interview, with, one, of, the, newspapers, there, and, it, came, up, of, course, it, was, not, a, sudden, suggestion, i, did, have, a, meeting, among, my, staff, and, related, ministries, and, i, talked, about, this, in, detail, before, i, came, to, the, united, states, we, have, a, new, administration, in, korea, and, we, haven't, yet, to, begun, dialogue, with, the, ns, inter, korean, dialogue, there, is, a, need, for, us, to, have, dialogue, all, ...]","[i, have, a, for, and, the, have, made, many, through, the, and, the, of, the, as, for, up, a, in, both, and, what, are, some, of, the, up, if, you, do, have, any, up, and, do, you, have, any, of, a, with, at, an]",0.511628,"[the, is, not, something, that, we, between, ourselves, during, the, in, when, i, was, in, i, had, an, with, one, of, the, there, and, it, up, of, it, was, not, a, i, have, a, among, my, and, and, i, about, this, in, detail, before, i, to, the, we, have, a, in, and, we, yet, to, with, the, there, is, a, for, us, to, have, all, the, up, until, now, we, had, whenever, the, and, then, it, would, however, should, be, on, and, and, so, with

In [ ]:
display(
    train_with_stop[train_with_stop["clarity_label"] == "Clear Reply"][cols_view].head(1)
)


,interview_question,interview_answer,q_tokens,a_tokens,q_stop_tokens,q_stop_ratio,a_stop_tokens,a_stop_ratio,stop_ratio_diff,clarity_label,evasion_label
0,Q. Of the Biden administration. And accused the United States of containing China while pushing for diplomatic talks.How would you respond to that? And do you think President Xi is being sincere about getting the relationship back on track as he bans Apple in China?,"Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues.And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things.And so, really, what this trip was aboutit was less about containing China. I don't want to contain China. I just want to make sure that we have a relationship with China that is on the up and up, squared away, everybody knows what it's all about. And one of the ways you do that is, you make sure that we are talking about the same things.And I think that one of the things we've doneI've tried to do, and I've talked with a number of my staff about this for the last, I guess, 6 monthsis, we have an opportunity to strengthen alliances around the world to maintain stability.That's what this trip was all about: having India cooperate much more with the United States, be closer with the United States, Vietnam being closer with the United States. It's not about containing China; it's about having a stable base, a stable base in the Indo-Pacific.And it'sfor example, when I was spending a lot of time talking with President Xi, he asked why we were doingwhy was I going to have the Quad, meaning Australia, India, Japan, and the United States? And I said, To maintain stability. It's not about isolating China. It's about making sure the rules of the roadeverything from airspace and space in the ocean isthe international rules of the road are abided by.And soand I hope thatI think that Prime Minister XiI mean, Xi has somesome difficulties right now. All countries end up with difficulties, and he had some economic difficulties he's working his way through. I want to see China succeed economically, but I want to see them succeed by the rules.The next question was to Bloomberg.","[q, of, the, biden, administration, and, accused, the, united, states, of, containing, china, while, pushing, for, diplomatic, talks, how, would, you, respond, to, that, and, do, you, think, president, xi, is, being, sincere, about, getting, the, relationship, back, on, track, as, he, bans, apple, in, china]","[well, look, first, of, all, thei, am, sincere, about, getting, the, relationship, right, and, one, of, the, things, that, is, going, on, now, is, china, is, beginning, to, change, some, of, the, rules, of, the, game, in, terms, of, trade, and, other, issues, and, so, one, of, the, things, we, talked, about, for, example, is, that, they're, now, talking, about, making, sure, that, no, chineseno, one, in, the, chinese, government, can, use, a, western, cell, phone, those, kinds, of, things, and, so, really, what, this, trip, was, aboutit, was, less, about, containing, china, i, don't, want, to, contain, china, i, ...]","[of, the, and, the, of, while, for, how, would, you, to, that, and, do, you, is, being, sincere, about, the, back, on, as, he, in]",0.543478,"[well, first, of, all, am, sincere, about, the, and, one, of, the, that, is, on, now, is, is, to, some, of, the, of, the, in, of, and, other, and, so, one, of, the, we, about, for, is, that, now, about, that, no, one, in, the, can, a, those, of, and, so, what, this, was, was, less, about, i, to, i, to, that, we, have, a, with, that, is, on, the, up, and, up, what, all, about, and, one, of, the, you, do, that, is, you, that, we, are, about, the, same, and, i, that, one, of, the, 

In [ ]:
display(
    train_with_stop[train_with_stop["clarity_label"] == "Ambivalent"][cols_view].head(1)
)


,interview_question,interview_answer,q_tokens,a_tokens,q_stop_tokens,q_stop_ratio,a_stop_tokens,a_stop_ratio,stop_ratio_diff,clarity_label,evasion_label
1,Q. Of the Biden administration. And accused the United States of containing China while pushing for diplomatic talks.How would you respond to that? And do you think President Xi is being sincere about getting the relationship back on track as he bans Apple in China?,"Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues.And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things.And so, really, what this trip was aboutit was less about containing China. I don't want to contain China. I just want to make sure that we have a relationship with China that is on the up and up, squared away, everybody knows what it's all about. And one of the ways you do that is, you make sure that we are talking about the same things.And I think that one of the things we've doneI've tried to do, and I've talked with a number of my staff about this for the last, I guess, 6 monthsis, we have an opportunity to strengthen alliances around the world to maintain stability.That's what this trip was all about: having India cooperate much more with the United States, be closer with the United States, Vietnam being closer with the United States. It's not about containing China; it's about having a stable base, a stable base in the Indo-Pacific.And it'sfor example, when I was spending a lot of time talking with President Xi, he asked why we were doingwhy was I going to have the Quad, meaning Australia, India, Japan, and the United States? And I said, To maintain stability. It's not about isolating China. It's about making sure the rules of the roadeverything from airspace and space in the ocean isthe international rules of the road are abided by.And soand I hope thatI think that Prime Minister XiI mean, Xi has somesome difficulties right now. All countries end up with difficulties, and he had some economic difficulties he's working his way through. I want to see China succeed economically, but I want to see them succeed by the rules.The next question was to Bloomberg.","[q, of, the, biden, administration, and, accused, the, united, states, of, containing, china, while, pushing, for, diplomatic, talks, how, would, you, respond, to, that, and, do, you, think, president, xi, is, being, sincere, about, getting, the, relationship, back, on, track, as, he, bans, apple, in, china]","[well, look, first, of, all, thei, am, sincere, about, getting, the, relationship, right, and, one, of, the, things, that, is, going, on, now, is, china, is, beginning, to, change, some, of, the, rules, of, the, game, in, terms, of, trade, and, other, issues, and, so, one, of, the, things, we, talked, about, for, example, is, that, they're, now, talking, about, making, sure, that, no, chineseno, one, in, the, chinese, government, can, use, a, western, cell, phone, those, kinds, of, things, and, so, really, what, this, trip, was, aboutit, was, less, about, containing, china, i, don't, want, to, contain, china, i, ...]","[of, the, and, the, of, while, for, how, would, you, to, that, and, do, you, is, being, sincere, about, the, back, on, as, he, in]",0.543478,"[well, first, of, all, am, sincere, about, the, and, one, of, the, that, is, on, now, is, is, to, some, of, the, of, the, in, of, and, other, and, so, one, of, the, we, about, for, is, that, now, about, that, no, one, in, the, can, a, those, of, and, so, what, this, was, was, less, about, i, to, i, to, that, we, have, a, with, that, is, on, the, up, and, up, what, all, about, and, one, of, the, you, do, that, is, you, that, we, are, about, the, same, and, i, that, one, of, the, 

In [ ]:
display(
    train_with_stop[train_with_stop["clarity_label"] == "Ambivalent"][cols_view].head(1)
)


In [ ]:
def print_example_vertical(df, row_idx: int):
    row = df.iloc[row_idx]

    print(f"=== INDEX: {row_idx} ===\n")

    print("InterviewQuestion:")
    print(row["interview_question"])
    print("\nInterviewAnswer:")
    print(row["interview_answer"])

    print("\nQuestionTokens:")
    print(row["q_tokens"])

    print("\nQuestionStopTokens:")
    print(row["q_stop_tokens"])
    print("QuestionStopRatio:", row["q_stop_ratio"])

    print("\nAnswerTokens:")
    print(row["a_tokens"])

    print("\nAnswerStopTokens:")
    print(row["a_stop_tokens"])
    print("\nAnswerStopRatio:", row["a_stop_ratio"])

    print("\nStopRatioDifference (Answer - Question):", row["stop_ratio_diff"])

    print("\nClarityLabel:", row["clarity_label"])
    print("\nEvasionLabel:", row["evasion_label"])

print_example_vertical(train_with_stop, 0)      # 0. örnek
# print_example_vertical(train_with_stop, 123)  # başka index denemek için


=== INDEX: 0 ===

InterviewQuestion:
Q. Of the Biden administration. And accused the United States of containing China while pushing for diplomatic talks.How would you respond to that? And do you think President Xi is being sincere about getting the relationship back on track as he bans Apple in China?

InterviewAnswer:
Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues.And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things.And so, really, what this trip was aboutit was less about containing China. I don't want to contain China. I just want to make sure that we have a relationship with China that is on the up and up, squared away, everybody knows what it's all about. And one

In [28]:
import re
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# ---------- 1) Basic setup ----------
STOP_SK = set(ENGLISH_STOP_WORDS)

NEGATIONS = {
    "no", "not", "never", "none", "nobody", "nothing", "nowhere",
    "neither", "nor", "cannot", "cant", "can't",
    "dont", "don't", "doesnt", "doesn't", "didnt", "didn't",
    "isnt", "isn't", "arent", "aren't", "wasnt", "wasn't",
    "werent", "weren't", "won't", "wouldnt", "wouldn't",
    "shouldnt", "shouldn't", "couldnt", "couldn't"
}

HEDGES = {
    "maybe", "perhaps", "probably", "possibly",
    "apparently", "seem", "seems", "seemed",
    "appear", "appears", "appeared",
    "roughly", "around", "about",
    "sort", "kinda", "kind", "somewhat",
    "guess", "think", "believe", "suggest", "suggests",
    "likely", "unlikely"
}

def tokenize(text: str):
    if not isinstance(text, str):
        return []
    text = text.lower()
    return re.findall(r"[a-z']+", text)

def stopword_ratio(tokens, stop_set=STOP_SK):
    if not tokens:
        return 0.0
    stop_count = sum(1 for t in tokens if t in stop_set)
    return stop_count / len(tokens)

def count_and_ratio(tokens, vocab: set):
    if not tokens:
        return 0, 0.0
    cnt = sum(1 for t in tokens if t in vocab)
    return cnt, cnt / len(tokens)


# ---------- 2) Build feature dataframe from RAW train ----------
train_feats = train_raw.copy()

# tokens
train_feats["q_tokens"] = train_feats["interview_question"].apply(tokenize)
train_feats["a_tokens"] = train_feats["interview_answer"].apply(tokenize)

# stopword features
train_feats["q_stop_tokens"] = train_feats["q_tokens"].apply(
    lambda toks: [t for t in toks if t in STOP_SK]
)
train_feats["a_stop_tokens"] = train_feats["a_tokens"].apply(
    lambda toks: [t for t in toks if t in STOP_SK]
)

# count + ratio
train_feats["q_stop_count"] = train_feats["q_stop_tokens"].apply(len)
train_feats["a_stop_count"] = train_feats["a_stop_tokens"].apply(len)

train_feats["q_stop_ratio"] = train_feats["q_tokens"].apply(stopword_ratio)
train_feats["a_stop_ratio"] = train_feats["a_tokens"].apply(stopword_ratio)
train_feats["stop_ratio_diff"] = train_feats["a_stop_ratio"] - train_feats["q_stop_ratio"]

# negation features
q_neg = train_feats["q_tokens"].apply(lambda toks: count_and_ratio(toks, NEGATIONS))
a_neg = train_feats["a_tokens"].apply(lambda toks: count_and_ratio(toks, NEGATIONS))

train_feats["q_neg_count"]  = q_neg.apply(lambda x: x[0])
train_feats["q_neg_ratio"]  = q_neg.apply(lambda x: x[1])
train_feats["a_neg_count"]  = a_neg.apply(lambda x: x[0])
train_feats["a_neg_ratio"]  = a_neg.apply(lambda x: x[1])
train_feats["neg_ratio_diff"] = train_feats["a_neg_ratio"] - train_feats["q_neg_ratio"]

# hedge features
q_hedge = train_feats["q_tokens"].apply(lambda toks: count_and_ratio(toks, HEDGES))
a_hedge = train_feats["a_tokens"].apply(lambda toks: count_and_ratio(toks, HEDGES))

train_feats["q_hedge_count"] = q_hedge.apply(lambda x: x[0])
train_feats["q_hedge_ratio"] = q_hedge.apply(lambda x: x[1])
train_feats["a_hedge_count"] = a_hedge.apply(lambda x: x[0])
train_feats["a_hedge_ratio"] = a_hedge.apply(lambda x: x[1])
train_feats["hedge_ratio_diff"] = train_feats["a_hedge_ratio"] - train_feats["q_hedge_ratio"]

# ---------- 3) Quick preview ----------
cols_view = [
    "interview_question",
    "interview_answer",
    "clarity_label",
    "evasion_label",
    "q_stop_count", "q_stop_ratio",
    "a_stop_count", "a_stop_ratio", "stop_ratio_diff",
    "q_neg_count", "q_neg_ratio",
    "a_neg_count", "a_neg_ratio", "neg_ratio_diff",
    "q_hedge_count", "q_hedge_ratio",
    "a_hedge_count", "a_hedge_ratio", "hedge_ratio_diff",
]

print("train_feats shape:", train_feats.shape)
display(train_feats[cols_view].head(5))


train_feats shape: (3448, 39)


,interview_question,interview_answer,clarity_label,evasion_label,q_stop_count,q_stop_ratio,a_stop_count,a_stop_ratio,stop_ratio_diff,q_neg_count,q_neg_ratio,a_neg_count,a_neg_ratio,neg_ratio_diff,q_hedge_count,q_hedge_ratio,a_hedge_count,a_hedge_ratio,hedge_ratio_diff
0,Q. Of the Biden administration. And accused the United States of containing China while pushing for diplomatic talks.How would you respond to that? And do you think President Xi is being sincere about getting the relationship back on track as he bans Apple in China?,"Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues.And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things.And so, really, what this trip was aboutit was less about containing China. I don't want to contain China. I just want to make sure that we have a relationship with China that is on the up and up, squared away, everybody knows what it's all about. And one of the ways you do that is, you make sure that we are talking about the same things.And I think that one of the things we've doneI've tried to do, and I've talked with a number of my staff about this for the last, I guess, 6 monthsis, we have an opportunity to strengthen alliances around the world to maintain stability.That's what this trip was all about: having India cooperate much more with the United States, be closer with the United States, Vietnam being closer with the United States. It's not about containing China; it's about having a stable base, a stable base in the Indo-Pacific.And it'sfor example, when I was spending a lot of time talking with President Xi, he asked why we were doingwhy was I going to have the Quad, meaning Australia, India, Japan, and the United States? And I said, To maintain stability. It's not about isolating China. It's about making sure the rules of the roadeverything from airspace and space in the ocean isthe international rules of the road are abided by.And soand I hope thatI think that Prime Minister XiI mean, Xi has somesome difficulties right now. All countries end up with difficulties, and he had some economic difficulties he's working his way through. I want to see China succeed economically, but I want to see them succeed by the rules.The next question was to Bloomberg.",Clear Reply,Explicit,25,0.543478,202,0.551913,0.008434,0,0.000000,4,0.010929,0.010929,2,0.043478,16,0.043716,0.000238
1,Q. Of the Biden administration. And accused the United States of containing China while pushing for diplomatic talks.How would you respond to that? And do you think President Xi is being sincere about getting the relationship back on track as he bans Apple in China?,"Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues.And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things.And so, really, what this trip was aboutit was less about containing China. I don't want to contain China. I just want to make sure that we have a relationship with China that is on the up and up, squared away, everybody knows what it's all about. And one of the ways you do that is, you make sure that we are talking about the same things.And I think that one of the things we've doneI've tried to do, and I've talked with a number of my staff about this for the last, I guess, 6 monthsis, we have an opportunity to strengthen alliances around the world to maintain stability.That's what this trip was all about: having India cooperate m